# Introduction to FEniCS
In the previous lecture you learnt how to assemble and solve a system of equations for a Poisson equation on line subdivided into $N$ equally large line segments.
However, in the real world, we will rarely work on so simple geometries, and will therefore find a higher level abstraction for representing these kinds of problems.

In this and the following lectures, we will use [FEniCS](https://fenicsproject.org/download/), a collection of software designed for the automated solution of partial differential equations using the finite element method. It handles a lot of the busywork involved for you, and more or less automates everything except the mathematical derivation of the problem (strong  and weak formulations). 
The community around the FEniCS project is very active and a lot of useful information and demo files can be found at the FEniCS Discourse (https://fenicsproject.discourse.group/). I recommend each of you to download the free tutorial book "Solving PDE's in Python - The FEniCS Tutorial I" (https://fenicsproject.org/tutorial/). It covers most of the Python functionality of FEniCS and is a good starting for any one who wants to use FEniCS.

Further tutorials on FEniCS can be found at https://jsdokken.com/dolfinx-tutorial/, https://jsdokken.com/FEniCS-workshop/ and https://docs.fenicsproject.org/dolfinx/v0.10.0.post5/python/demos.html.

## Reimplementing the FEM intro problem on a grid with spatially varying cells
In this demo, we will consider a problem similar to [L01_FEM_intro](./L01_FEM_intro), but we will use a grid where the points are not equally spaced.
We start by importing the modules we require for generating our mesh

In [3]:

from mpi4py import MPI
import numpy as np
import basix

The modules we have import so far is {py:mod}`mpi4py`, a Python module for handling communication across different processes (CPUs) on a system. In this lecture we will not focus on how this works. Furthermore, we import {py:mod}`numpy` for usage of numerical arrays, and {py:mod}`basix` which is a finite element library.
Next, we generate two arrays, one called `nodes` which contains the points of our mesh and `cells` which is a 2D array where the i-th row describes the position of the vertices of cell `i` in `nodes`

In [4]:
cells = np.array([[0,1], [1,2], [2, 3], [3, 4], [4, 6], [6,5]], dtype=np.int64)
nodes = np.array([[0.1], [0.2], [0.4], [0.5],[0.6], [1.], [0.8]], dtype=np.float64)

Furthermore, as each cell is not the same size, we cannot pre-compute the integrals in the matrix `A` for all cells. A key-ingredient in the finite element method is to **pull back** all integrals to a reference cell, meaning that the integral

$$ \int_a^b f(x)~\mathrm{d}x = \int_0^1 f(F(\hat x))\vert b - a \vert~\mathrm{d}\hat x,$$
where $F(\hat x):[0, 1]\mapsto[a, b]=a + \hat x (b-a)$.

More generally, if there exists a mapping: $F:K_{ref}\mapsto K$ then we can write the integral

$$ \int_K f(x)~\mathrm{d}x = \int_{K_{ref}} f(F(\hat x))\vert \mathrm{det} J\vert~\mathrm{d}\hat x,$$

where $J$ is the Jacobian of the mapping $F$, i.e. $J = \frac{\partial F}{\partial \hat x}$. This is generally known as the method of mappings.

We choose to describe any cell $K$ by a finite element, i.e.

$$
F_i(\hat x) = \sum_{k=0}^N \mathbf{p}_{i,k} \phi_k(\hat x)
$$
where $\mathbf{p}_{i,k}$ is the $k$ node of the $i$ th cell, where $\phi_k$ is the $k$-basis function of the chosen space. For straight intervals, we can choose a first order Lagrange element. To be general, we use basix to create this element.
To create a {py:class}`dolfinx.mesh.Mesh` object, we use the function {py:func}`dolfinx.mesh.create_mesh` to collect all the data as a single object.

In [5]:
from dolfinx.mesh import create_mesh
c_el = basix.ufl.element("Lagrange", "interval", 1, shape=(nodes.shape[1], ))
mesh = create_mesh(MPI.COMM_SELF, cells=cells, e=c_el, x=nodes)


### Evaluating a finite element at some points
With the basix element, we can evaluate the basis functions at various points.
We use the {py:meth}`tabulate<basix.finite_element.FiniteElement.tabulate>` function to get the $k$-th order derivatives of the basis function.

In [6]:
points = np.array([[0],[0.4],[1]]) # Pick some points in the reference interval
values = c_el.tabulate(0, points) # Tabulate the basis functions (0th order derivatives) at these points
for val, point in zip(values, points):
    print(f"Basis functions at {point}: {val}")

Basis functions at [0.]: [[[ 1.00000000e+00 -3.82105486e-17]]

 [[ 6.00000000e-01  4.00000000e-01]]

 [[ 5.79375858e-17  1.00000000e+00]]]


## Exercise

```{exercise} Compute physical coordiantes
:label: l12-mapping-F
Compute the coordinate of the midpoint of each cell by using the mapping $F_i$
```

Go to {ref}`solution <sol-l12-mapping-F>`.


```{exercise} Compute physical coordinates of a tetrahedron
:label: l12-mapping-tetra-F
Define a tetrahedron by 4 vertices, compute from a reference tetrahedron to physical space.
```

```{dropdown} Hint
To get some points within a reference tetrahedron, you can use {py:func}`basix.create_lattice`, for instance 
```python
reference_points = basix.create_lattice(
    basix.CellType.tetrahedron, 8, basix.LatticeType.gll, exterior=False, method=basix.LatticeSimplexMethod.warp
)
```
Go to {ref}`solution <sol-l12-mapping-tetra-F>`.


#### Exercise
1. Compute the map $F(\hat x)$ for the first and third cell of the mesh.
2. Compute the determinant of the map using `c_el`.
3. Create a single element triangular mesh with vertices $(0,0), (2, 0), (0, 1), (2, 1)$. Repeat exercise 1 and 2 on this mesh.

## Below are things that will be embedded

In [ ]:



# Note that we have chosen to use intervals embedded in R^2, i.e. a manifold surface, thus
# the geometric dimesion of the mesh is 2, while the topological dimension is 1 (intervals)

gdim = mesh.geometry.dim
tdim = mesh.topology.dim

# The number of cells owned by this process can be extracted from the index map
num_cells = mesh.topology.index_map(tdim).size_local

# We create the function space for our unknown, pick whatever degree you'd like
degree = 3
el = basix.ufl.element("Lagrange", mesh.basix_cell(), degree)
V = dolfinx.fem.functionspace(mesh, el)

# To get exact integration of straight intervals for a stiffness matrix, we pick a
# sufficiently high quadrature degree

q_deg = 2*(degree - 1)
q_pts, q_weights = basix.make_quadrature(mesh.basix_cell(), q_deg)

# Next, we will tabulate the basis functions required to compute the jacobian
# and gradientes on the space chosen for the unknown on the reference element.

# Compute Jacobians on reference elements
# The first entry to tabulate is the number of derivatives (we skip the basis values phi_i(x)...
# by using [1:])
coordinate_basis_derivatives = c_el.sub_elements[0].tabulate(1, q_pts)[1:]

# Compute basis functions and derivatives on reference element
# Similar procedure to what we saw above
el_basis_derivatives = el.tabulate(1, q_pts)[1:] # shape: (derivative direction, num_pts, num_basis_funcs)
num_basis_funcs = el_basis_derivatives.shape[2]

# Create local matrix to populate in assembly
A_loc = np.zeros((num_basis_funcs, num_basis_funcs), dtype=np.float64)

# Create a petsc matrix where we allocate the potential non-zero entries by looping through the dofmap and inserting them 
# into a sparsity pattern
pattern = dolfinx.fem.SparsityPattern(mesh.comm, [V.dofmap.index_map, V.dofmap.index_map], [V.dofmap.index_map_bs, V.dofmap.index_map_bs])
for i in range(num_cells):
    local_dofs = V.dofmap.cell_dofs(i)
    pattern.insert(local_dofs, local_dofs)
pattern.finalize()

# Create matrix
A = dolfinx.cpp.la.petsc.create_matrix(mesh.comm, pattern)
A.zeroEntries()

# Assemble over cells local to process
for i in range(num_cells):
    # Zero local tensor
    A_loc[:,:] = 0
    
    # Extract cell nodal coordinates
    cell_node_indices = mesh.geometry.dofmap[i]
    local_mesh_geometry = mesh.geometry.x[cell_node_indices, :mesh.geometry.dim]

    # Compute jacobian for cell
    dphi_dxi = []
    for j in range(tdim):
        dphi_dxi.append(coordinate_basis_derivatives[j] @ local_mesh_geometry)
    jacobian = np.transpose(np.hstack(dphi_dxi).reshape(q_pts.shape[0], tdim, gdim), (0, 2, 1))

    # Loop over quadrature points
    for j in range(q_pts.shape[0]): 
        # Since we are working on a manifold we need to compute the Moore-Penrose pseudo-inverse and
        # Gram determinant rather than the standard inverse and determinant

        # Moore-Penrose pseudo-inverse   
        jacobian_T = jacobian[j].T
        jacobian_pinv = np.linalg.pinv(jacobian_T)
        # Gram determinant
        jac_det = np.sqrt(np.linalg.det(jacobian_T@jacobian[j]))

        # Insert into local tensor
        for k in range(num_basis_funcs):
            for l in range(num_basis_funcs):
                dphi_k = el_basis_derivatives[:, j, k]
                dphi_l = el_basis_derivatives[:, j, l]
                A_loc[k, l] += q_weights[j] * np.dot(jacobian_pinv @  dphi_k, jacobian_pinv @ dphi_l) * jac_det
    
    # Insert local matrix into global matrix
    cell_dofs = V.dofmap.cell_dofs(i)
    A.setValuesLocal(cell_dofs, cell_dofs, A_loc, PETSc.InsertMode.ADD_VALUES)
# Accumulate contributions from all processors
A.assemble()


# Compare with solution made with UFL
u = ufl.TrialFunction(V)
v = ufl.TestFunction(V)
A_ref = dolfinx.fem.petsc.assemble_matrix(dolfinx.fem.form(ufl.inner(ufl.grad(u), ufl.grad(v))*ufl.dx))
A_ref.assemble()

if MPI.COMM_WORLD.size == 0:
    np.testing.assert_allclose(A_ref.to_dense(), A[:,:])

D = A - A_ref
PETSc.Sys.Print(A.norm(2), A_ref.norm(2), D.norm(2))
assert np.isclose(D.norm(2), 0)